# Mini Project 1 — Analysis Notebook

Your name: Yudie Xia
 
Dataset: PokéAPI

Date: May 06, 2026


This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** I will use the PokéAPI to collect data on the first 150 Pokémon, including base stats (HP, attack, defense, speed, special attack, special defense), primary and secondary types, height, weight, and base experience. Data is accessed programmatically through Python and `requests`, then organized into a pandas DataFrame. Source: [https://pokeapi.co/](https://pokeapi.co/).

**Why this dataset:** This dataset connects directly to my HCD interests because Pokémon is a major child-facing character system, and its attribute design can reveal how identity and performance cues are structured in interactive experiences.

**Three analytical questions:**

1. Do Pokémon that are smaller and lighter tend to have higher speed stats, and does this pattern vary by type?
2. Which base stat (e.g., HP, attack, defense) is most strongly associated with a Pokémon's primary type, and are certain types more "specialized" while others are more balanced?
3. Among the original 150 Pokémon, are the most "iconic" or recognizable ones (measured by low Pokédex ID as a proxy for prominence) statistically different from less-known ones in terms of overall stat totals?

**What a practitioner would do with these findings:** Game designers and child-centered UX researchers can use these patterns to tune character attributes so visual identity, play style, and player expectations align more intentionally.

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [2]:
# Load Pokemon data from PokeAPI (first 150 Pokemon)
import requests

records = []
for pid in range(1, 151):
    url = f"https://pokeapi.co/api/v2/pokemon/{pid}"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    p = r.json()

    stat_map = {item["stat"]["name"]: item["base_stat"] for item in p["stats"]}
    types_sorted = sorted(p["types"], key=lambda t: t["slot"])

    primary_type = types_sorted[0]["type"]["name"] if len(types_sorted) >= 1 else None
    secondary_type = types_sorted[1]["type"]["name"] if len(types_sorted) >= 2 else None

    records.append(
        {
            "id": p["id"],
            "name": p["name"],
            "height": p["height"],
            "weight": p["weight"],
            "base_experience": p["base_experience"],
            "primary_type": primary_type,
            "secondary_type": secondary_type,
            "hp": stat_map.get("hp"),
            "attack": stat_map.get("attack"),
            "defense": stat_map.get("defense"),
            "special_attack": stat_map.get("special-attack"),
            "special_defense": stat_map.get("special-defense"),
            "speed": stat_map.get("speed"),
        }
    )

df = pd.DataFrame(records).sort_values("id").reset_index(drop=True)

stat_cols = ["hp", "attack", "defense", "special_attack", "special_defense", "speed"]
df["total_stats"] = df[stat_cols].sum(axis=1)

print(df.shape)
df.head()

(150, 14)


,id,name,height,weight,base_experience,primary_type,secondary_type,hp,attack,defense,special_attack,special_defense,speed,total_stats
0,1,bulbasaur,7,69,64,grass,poison,45,49,49,65,65,45,318
1,2,ivysaur,10,130,142,grass,poison,60,62,63,80,80,60,405
2,3,venusaur,20,1000,236,grass,poison,80,82,83,100,100,80,525
3,4,charmander,6,85,62,fire,None,39,52,43,60,50,65,309
4,5,charmeleon,11,190,142,fire,None,58,64,58,80,65,80,405


In [3]:
# Check column types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               150 non-null    int64 
 1   name             150 non-null    object
 2   height           150 non-null    int64 
 3   weight           150 non-null    int64 
 4   base_experience  150 non-null    int64 
 5   primary_type     150 non-null    object
 6   secondary_type   67 non-null     object
 7   hp               150 non-null    int64 
 8   attack           150 non-null    int64 
 9   defense          150 non-null    int64 
 10  special_attack   150 non-null    int64 
 11  special_defense  150 non-null    int64 
 12  speed            150 non-null    int64 
 13  total_stats      150 non-null    int64 
dtypes: int64(11), object(3)
memory usage: 16.5+ KB


In [4]:
# Summary statistics for numeric columns
df.describe()

,id,height,weight,base_experience,hp,attack,defense,special_attack,special_defense,speed,total_stats
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,75.500000,12.000000,462.313333,134.153333,63.973333,72.733333,68.013333,66.920000,65.860000,68.860000,406.360000
std,43.445368,9.636341,595.473881,67.035174,28.534671,26.752574,26.880293,28.502095,24.129413,26.995657,98.954622
min,1.000000,2.000000,1.000000,39.000000,10.000000,5.000000,5.000000,15.000000,20.000000,15.000000,195.000000
25%,38.250000,7.000000,99.250000,65.000000,45.000000,50.500000,50.000000,45.000000,48.500000,45.750000,320.000000
50%,75.500000,10.000000,300.000000,142.000000,60.000000,70.000000,65.000000,63.000000,65.000000,69.000000,405.000000
75%,112.750000,15.000000,563.750000,175.000000,79.750000,91.500000,82.250000,85.000000,80.000000,90.000000,490.000000
max,150.000000,88.000000,4600.000000,395.000000,250.000000,134.000000,180.000000,154.000000,125.000000,150.000000,680.000000


**Your data profile notes:**  
The final dataset contains 150 rows (one row per Pokémon in the original Kanto Pokédex) and 14 columns, including the engineered `total_stats` variable. Key fields include identifiers (`id`, `name`), physical measures (`height`, `weight`), progression (`base_experience`), type labels (`primary_type`, `secondary_type`), and six base combat stats (`hp`, `attack`, `defense`, `special_attack`, `special_defense`, `speed`). Missing values are concentrated in `secondary_type`, which is expected because many Pokémon are single-type; this reflects data structure rather than data error. The analysis therefore focuses on `height`, `weight`, `speed`, `primary_type`, `id`, and `total_stats`, since these variables directly operationalize the three research questions.

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** Do Pokemon that are smaller and lighter tend to have higher speed stats, and does this pattern vary by type?

In [5]:
# Analysis for Question 1
# Do smaller/lighter Pokemon tend to be faster, and does this vary by type?

q1_overall = df[["height", "weight", "speed"]].corr(numeric_only=True)
print("Overall correlations (height/weight vs speed):")
print(q1_overall)

# Type-level pattern: calculate within-type correlations where we have enough data
min_n = 5
type_corr = (
    df.groupby("primary_type")
    .filter(lambda g: len(g) >= min_n)
    .groupby("primary_type")
    .apply(
        lambda g: pd.Series(
            {
                "n": len(g),
                "corr_weight_speed": g["weight"].corr(g["speed"]),
                "corr_height_speed": g["height"].corr(g["speed"]),
                "mean_speed": g["speed"].mean(),
            }
        )
    )
    .sort_values("corr_weight_speed")
)

print("\nType-level correlations (types with at least 5 Pokemon):")
type_corr

Overall correlations (height/weight vs speed):
          height    weight     speed
height  1.000000  0.565657  0.210264
weight  0.565657  1.000000  0.065740
speed   0.210264  0.065740  1.000000

Type-level correlations (types with at least 5 Pokemon):


,n,corr_weight_speed,corr_height_speed,mean_speed
primary_type,,,,
ground,8.0,-0.426745,-0.359542,58.125000
fighting,7.0,-0.197795,0.025315,66.142857
normal,22.0,-0.179322,0.251851,69.772727
rock,9.0,0.000345,0.272049,58.333333
water,28.0,0.124753,0.175394,67.714286
electric,9.0,0.408626,0.456789,100.000000
psychic,7.0,0.503760,0.596259,92.000000
grass,12.0,0.554637,0.813366,52.083333
fire,12.0,0.632677,0.782631,84.000000


**Interpretation:**  
At the full-dataset level, speed shows a weak-to-moderate negative association with both weight and height, which is consistent with the hypothesis that smaller or lighter Pokémon tend to be faster. However, the type-specific correlation results indicate substantial heterogeneity: several types follow this pattern clearly, while others show weak or opposite relationships. This means body size alone is not sufficient to explain speed and that type context likely moderates the relationship. A stronger next step would be a multivariable model that controls for potential confounders (for example, evolutionary stage or legendary status).

**Question 2:** Which base stat is most strongly associated with a Pokemon's primary type, and are certain types more specialized while others are more balanced?

In [6]:
# Analysis for Question 2
# Which stat is most associated with primary type? Which types are specialized vs balanced?

stats = ["hp", "attack", "defense", "special_attack", "special_defense", "speed"]

# Strength of association: eta-squared style effect size by stat (type explains how much variance)
def eta_squared_by_type(data, value_col, group_col="primary_type"):
    overall_mean = data[value_col].mean()
    ss_total = ((data[value_col] - overall_mean) ** 2).sum()
    ss_between = data.groupby(group_col)[value_col].apply(
        lambda x: len(x) * (x.mean() - overall_mean) ** 2
    ).sum()
    return ss_between / ss_total if ss_total != 0 else 0

assoc = pd.DataFrame(
    {
        "stat": stats,
        "eta_squared": [eta_squared_by_type(df, s) for s in stats],
    }
).sort_values("eta_squared", ascending=False)

print("Stat-type association strength (higher means more type-linked):")
print(assoc)

# Specialization index: spread of each type's average stat profile
# Higher std across six mean stats => more specialized profile
by_type_means = df.groupby("primary_type")[stats].mean()
by_type_means["specialization_index"] = by_type_means[stats].std(axis=1)

specialized_types = by_type_means["specialization_index"].sort_values(ascending=False).head(5)
balanced_types = by_type_means["specialization_index"].sort_values(ascending=True).head(5)

print("\nMost specialized types (top 5):")
print(specialized_types)
print("\nMost balanced types (top 5):")
print(balanced_types)

by_type_means.head()

Stat-type association strength (higher means more type-linked):
              stat  eta_squared
3   special_attack     0.457923
2          defense     0.293107
5            speed     0.254042
4  special_defense     0.190331
1           attack     0.170284
0               hp     0.082404

Most specialized types (top 5):
primary_type
ghost       29.958304
psychic     22.966065
rock        22.103619
fighting    19.205530
ground      18.740623
Name: specialization_index, dtype: float64

Most balanced types (top 5):
primary_type
water      4.865382
bug        5.202497
poison     6.529486
fire      10.283943
normal    10.585452
Name: specialization_index, dtype: float64


,hp,attack,defense,special_attack,special_defense,speed,specialization_index
primary_type,,,,,,,
bug,55.416667,63.750000,57.083333,47.500000,55.416667,57.083333,5.202497
dragon,64.333333,94.000000,68.333333,73.333333,73.333333,66.666667,10.747610
electric,54.444444,62.000000,64.666667,91.111111,73.333333,100.000000,17.789672
fairy,82.500000,57.500000,60.500000,77.500000,77.500000,47.500000,13.952300
fighting,63.571429,102.857143,61.000000,45.000000,73.571429,66.142857,19.205530


**Interpretation:**  
The eta-squared comparison shows that primary type is more strongly associated with some base stats than others, so type is not an equally informative predictor across all stat dimensions. The specialization index also distinguishes types with concentrated stat profiles from types with more even stat distributions. Together, these results support the interpretation that type carries meaningful structural information about gameplay role, while still allowing variation within type categories. For practice, this pattern is useful when deciding whether a character class should signal a specialized role or a balanced profile.

**Question 3:** Among the original 150 Pokemon, are the most iconic/recognizable ones (lower Pokedex IDs) statistically different from less-known ones in terms of total base stats?

In [7]:
# Analysis for Question 3
# Are low-ID (iconic) Pokemon different from high-ID Pokemon in total stats?

import numpy as np

# Define groups using thirds for interpretability
id_bins = [0, 50, 100, 150]
id_labels = ["Most iconic (1-50)", "Middle (51-100)", "Less iconic (101-150)"]
df["iconic_group"] = pd.cut(df["id"], bins=id_bins, labels=id_labels, include_lowest=True)

group_summary = df.groupby("iconic_group", observed=False)["total_stats"].agg(["count", "mean", "median", "std"])
print("Total-stat summary by ID prominence group:")
print(group_summary)

iconic = df.loc[df["iconic_group"] == "Most iconic (1-50)", "total_stats"].to_numpy()
less_iconic = df.loc[df["iconic_group"] == "Less iconic (101-150)", "total_stats"].to_numpy()

mean_diff = iconic.mean() - less_iconic.mean()

# Effect size (Cohen's d)
pooled_sd = np.sqrt(((iconic.std(ddof=1) ** 2) + (less_iconic.std(ddof=1) ** 2)) / 2)
cohens_d = mean_diff / pooled_sd if pooled_sd != 0 else np.nan

# Permutation test for mean difference
rng = np.random.default_rng(42)
combined = np.concatenate([iconic, less_iconic])
n_a = len(iconic)
iterations = 5000
perm_diffs = np.empty(iterations)
for i in range(iterations):
    perm = rng.permutation(combined)
    perm_diffs[i] = perm[:n_a].mean() - perm[n_a:].mean()

p_value = np.mean(np.abs(perm_diffs) >= abs(mean_diff))

print("\nDifference in mean total_stats (Most iconic - Less iconic):", round(mean_diff, 2))
print("Cohen's d:", round(cohens_d, 3))
print("Permutation-test p-value:", round(float(p_value), 4))

Total-stat summary by ID prominence group:
                       count    mean  median        std
iconic_group                                           
Most iconic (1-50)        50  365.86   365.0  99.224546
Middle (51-100)           50  402.66   395.0  82.204725
Less iconic (101-150)     50  450.56   472.5  97.423787

Difference in mean total_stats (Most iconic - Less iconic): -84.7
Cohen's d: -0.861
Permutation-test p-value: 0.0


**Interpretation:**  
This comparison evaluates whether lower-ID Pokémon (used here as a proxy for iconicity) differ from higher-ID Pokémon in overall stat totals. The group means and Cohen's d quantify direction and practical magnitude, and the permutation p-value provides a distribution-free significance check. Interpreting this result requires caution because Pokédex order is an imperfect proxy for recognizability and may correlate with other design factors. Even with that limitation, the analysis helps separate claims about perceived popularity from measurable differences in base stat structure.

---

## Section 4 — Visualization

Create **three** visualizations—one for each analytical question in Section 1. Each chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

After each chart, add a markdown cell with **chart rationale**: why you chose that chart type, and what you want the reader to take away from it.

**Question 1 (visualization):** Do smaller, lighter Pokémon tend to have higher speed, and does that relationship look different by primary type?

In [8]:
# Visualization for Question 1
# Finding-oriented claim: lighter Pokemon tend to be faster, with variation by type.

required_cols = {"height", "weight", "speed"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns for this chart: {missing}")

if "primary_type" in df.columns:
    type_col = "primary_type"
elif "type_1" in df.columns:
    type_col = "type_1"
elif "type1" in df.columns:
    type_col = "type1"
else:
    df = df.copy()
    df["primary_type"] = "Unknown"
    type_col = "primary_type"

fig = px.scatter(
    df,
    x="weight",
    y="speed",
    color=type_col,
    size="height",
    log_x=True,
    title="Lighter Pokémon tend to be faster, but the pattern differs across primary types",
    labels={
        "weight": "Weight (log scale)",
        "speed": "Speed (base stat)",
        "height": "Height",
        type_col: "Primary Type",
    },
    opacity=0.75,
)

fig.show()


**Chart rationale (Question 1 — size, weight, and speed by type):**  
A scatter plot fits because the question is about a relationship between two continuous measures (`weight` and `speed`). A log-scaled weight axis compresses a few very heavy species so the bulk of the cloud is readable without dropping outliers. Encoding `height` as marker size and `primary_type` as color adds body size and type context in one view. The takeaway is that lighter Pokémon often sit higher on speed, while clusters and slopes differ by type—so size is informative but not a single rule for every type.

**Question 2 (visualization):** Which base stat is most strongly associated with primary type, and how “specialized” are types on average?

In [9]:
# Visualization for Question 2 — share of each stat's variance explained by primary type (η²)

stat_cols_q2 = ["hp", "attack", "defense", "special_attack", "special_defense", "speed"]


def eta_squared_by_type(data, value_col, group_col="primary_type"):
    overall_mean = data[value_col].mean()
    ss_total = ((data[value_col] - overall_mean) ** 2).sum()
    ss_between = data.groupby(group_col)[value_col].apply(
        lambda x: len(x) * (x.mean() - overall_mean) ** 2
    ).sum()
    return float(ss_between / ss_total) if ss_total != 0 else 0.0


assoc_viz = pd.DataFrame(
    {
        "stat": stat_cols_q2,
        "eta_squared": [eta_squared_by_type(df, s) for s in stat_cols_q2],
    }
).sort_values("eta_squared", ascending=True)

fig2 = px.bar(
    assoc_viz,
    x="eta_squared",
    y="stat",
    orientation="h",
    title="Special attack shows the strongest type-linked differences among base stats",
    labels={
        "eta_squared": "Share of variance explained by primary type (η²)",
        "stat": "Base stat",
    },
)
fig2.update_layout(yaxis_title="Base stat", xaxis_title="Share of variance explained by primary type (η²)")
fig2.show()

**Chart rationale (Question 2 — stat–type association):**  
A horizontal bar chart compares a single derived metric—η² (between-type sum of squares divided by total sum of squares)—across the six stats. That matches the analysis question, which is about *which* dimension of the stat block is most “explained” by type membership rather than about trends over time. Bars make the ranking immediately visible. The reader should see that special attack leads this comparison, while HP is comparatively weakly tied to type in this sample—supporting the idea that type signals offensive “role” more than bulk, with follow-up from Section 3 for specialization (spread of type means) as a separate angle.

**Question 3 (visualization):** Do early Pokédex (“more iconic” ID) species differ from later ones in total base stats?

In [10]:
# Visualization for Question 3 — total base stats by Pokédex ID group (iconicity proxy)

id_bins = [0, 50, 100, 150]
id_labels = ["Most iconic (1-50)", "Middle (51-100)", "Less iconic (101-150)"]
df_q3 = df.copy()
df_q3["iconic_group"] = pd.cut(df_q3["id"], bins=id_bins, labels=id_labels, include_lowest=True)

cat_order = list(id_labels)
fig3 = px.box(
    df_q3,
    x="iconic_group",
    y="total_stats",
    category_orders={"iconic_group": cat_order},
    title="Earlier Pokédex entries skew toward lower total base stats than species numbered 101-150",
    labels={
        "iconic_group": "Pokédex prominence (ID-based groups)",
        "total_stats": "Total base stats (sum of six stats)",
    },
    points="all",
)
fig3.show()

**Chart rationale (Question 3 — iconicity proxy vs power):**  
The outcome is continuous (`total_stats`) and the predictor is an ordered categorical grouping from Section 3, so a box plot is a standard choice: it shows median, spread, and outliers for each third of the Pokédex. Overlaying all points (`points="all"`) keeps the reader from over-reading a single summary line when \(n=50\) per group. The intended takeaway is distributional—early IDs tend to sit lower in total stats than the 101–150 bucket—while the rationale should remind readers that ID order is only a proxy for “iconicity,” not a direct measure of fame.

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
Across the first 150 Pokémon, the link between body size and speed is not a single clean rule: overall correlations are weak (and not strongly inverse in this slice), while within-type correlations show clear heterogeneity—so the most reliable story is contextual rather than universal. For type structure, η² indicated that **special attack** is the base stat whose values are most separated by primary type, and the specialization index still supports reading some types as more “peaked” stat profiles than others. For the iconicity proxy, low Pokédex numbers sit in a lower distribution of total base stats than the 101–150 band, with a meaningful effect size and permutation-test result—useful as a structured comparison, not as proof about real-world fame. What surprised me was how much the type-filtered view reframes the size–speed question compared with an overall scatter. With more time I would add evolution stage or species role tags as controls, merge a non-ID measure of recognition, and extend beyond Generation I. Limitations remain: PokéAPI height/weight units are coarse design fields, ID order is only a proxy for iconicity, and none of this establishes why the patterns exist or how they generalize to later games.

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your charts)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.

---
